In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install wandb --quiet

## Config

Set `REPO_ROOT` to the absolute path of the repo on your Google Drive. All other paths are relative to it.

In [ ]:
import os
import sys
import json
import pickle as pkl
from dataclasses import asdict
from typing import Optional, Tuple

import torch
import wandb
from torch.utils.data import DataLoader

In [ ]:
REPO_ROOT = "/content/drive/MyDrive/RutishauserLab/AI4Sci/METHODS/DiffusionBANMap/"
CONFIG_FILE = "config/overfit_cfd.json"
RUN_NAME = "run1"
RUN_NOTES = ""
RESUME_CKPT_PATH = None
NUM_WORKERS = 2

# Paths stored in the pickle were created on a different machine.
# Set these to remap them to the current environment.
# Example: LOCAL_DATA_PREFIX = "G:\\My Drive"  →  COLAB_DATA_PREFIX = "/content/drive/MyDrive"
LOCAL_DATA_PREFIX = "G:\\My Drive"
COLAB_DATA_PREFIX = "/content/drive/MyDrive"

## Imports

In [ ]:
sys.path.insert(0, REPO_ROOT)

import importlib.metadata
if not hasattr(importlib.metadata, '_patched'):
    _original_version = importlib.metadata.version
    def _patched_version(name):
        try:
            return _original_version(name)
        except importlib.metadata.PackageNotFoundError:
            return 'unknown'
    importlib.metadata.version = _patched_version
    importlib.metadata._patched = True

from data_utils import FaceDataset  # noqa: F401 — needed for pickle
from models.beta_vae import BetaVAE
from scripts.config import TrainingConfig
from scripts.trainer import BetaVAETrainer
from utils import discover_device, count_model_params

## Load config & build model

In [ ]:
config_path = os.path.join(REPO_ROOT, CONFIG_FILE)
with open(config_path) as f:
    raw = json.load(f)

# Resolve dataset and output paths relative to REPO_ROOT
for key in ("train_dataset_path", "test_dataset_path", "output_dir"):
    if key in raw and raw[key]:
        raw[key] = os.path.join(REPO_ROOT, raw[key].lstrip("./"))

config = TrainingConfig(**raw)

device = torch.device(discover_device())
print(f"Device: {device}")

model = BetaVAE(
    input_channels=config.input_channels,
    input_height=config.input_height,
    input_width=config.input_width,
    latent_dim=config.latent_dim,
    hidden_dim=config.hidden_dim,
    beta=config.beta,
    device=device
)
count_model_params(model)

## Data

In [ ]:
def load_dataloader(path: Optional[str], shuffle: bool) -> Optional[DataLoader]:
    if not path:
        return None
    with open(path, "rb") as f:
        dataset = pkl.load(f)
    dataset.remap_paths(LOCAL_DATA_PREFIX, COLAB_DATA_PREFIX)
    loader = DataLoader(dataset, batch_size=config.batch_size, shuffle=shuffle, num_workers=NUM_WORKERS)
    print(f"{path}: {len(dataset)} images, {len(loader)} batches")
    return loader

train_dataloader = load_dataloader(config.train_dataset_path, shuffle=True)
val_dataloader   = load_dataloader(config.test_dataset_path,  shuffle=False)

## Wandb

In [ ]:
wandb.login(key="...")

wandb.init(
    project="diffusion_ban_map",
    name=f"{config.experiment_name}_{RUN_NAME}",
    notes=RUN_NOTES,
    config=asdict(config)
)

## Train

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)

trainer = BetaVAETrainer(
    model=model,
    optimizer=optimizer,
    scheduler=None,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    config=config,
    device=device
)

trainer.train(resume_ckpt_path=RESUME_CKPT_PATH)